# RefillCare — Phase 3: Leakage-Safe Feature Engineering & Supervised Dataset

## 1. Objective
Phase 3 formulates the supervised machine learning target ($y_i = t_{i+1} - t_i$) and engineers 36 backward-looking features strictly without lookahead data leakage.


## 2. Input Data Setup


In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np

# Universal workspace root and sys.path resolver
current_dir = Path(__file__).resolve().parent if "__file__" in locals() else Path.cwd()
project_root = current_dir.resolve()
while project_root.parent != project_root and not (project_root / "refillcare" / "__init__.py").exists():
    project_root = project_root.parent

if (project_root / "refillcare" / "__init__.py").exists() and str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Safe display helper for Jupyter and standalone environments
try:
    from IPython.display import display
except ImportError:
    display = print

# Safe matplotlib import
try:
    import matplotlib
    if "ipykernel" not in sys.modules:
        matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

def find_file(relative_path: str) -> Path:
    candidates = [
        project_root / relative_path,
        Path.cwd() / relative_path,
        Path("..") / relative_path,
        Path("../..") / relative_path,
    ]
    for c in candidates:
        if c.exists():
            return c.resolve()
    return candidates[0]

import json

train_path = find_file("data/refillcare/processed/train.parquet")
val_path = find_file("data/refillcare/processed/validation.parquet")
test_path = find_file("data/refillcare/processed/test.parquet")

train_df = pd.read_parquet(train_path)
val_df = pd.read_parquet(val_path)
test_df = pd.read_parquet(test_path)

print(f"Train set:      {len(train_df):,} rows ({train_df.invoice_date.min().date()} to {train_df.invoice_date.max().date()})")
print(f"Validation set: {len(val_df):,} rows ({val_df.invoice_date.min().date()} to {val_df.invoice_date.max().date()})")
print(f"Test set:       {len(test_df):,} rows ({test_df.invoice_date.min().date()} to {test_df.invoice_date.max().date()})")


## 3. Processing & Leakage Protection Rules
- **Target Formulation:** $y_i = \text{date}(i+1) - \text{date}(i)$.
- **Final Purchase Rule:** Event $N$ in every history has no known next purchase and is excluded from training targets.
- **Temporal Splitting:** Non-overlapping chronological windows (Train $\le$ 2026-04-30, Val: 2026-05 to 2026-06, Test: 2026-07 to 2026-08).
- **No Lookahead:** Historical expanding interval median, mean, and std use only intervals completed on or before purchase event $i$.


In [ ]:
# Display feature matrix columns
feature_cols = [
    "purchase_count_so_far", "days_since_first_purchase", "historical_interval_median",
    "avg_historical_quantity", "quantity_vs_avg_ratio", "purchase_month", "target_days_until_next_purchase"
]
display(train_df[feature_cols].head(8))


## 4. Results: Target Distribution & Baseline Benchmark


In [ ]:
# Visualize target days until next purchase across partitions
if plt is not None:
    plt.figure(figsize=(10, 4))
    plt.hist(train_df["target_days_until_next_purchase"].clip(upper=100), bins=35, color="#4b6584", alpha=0.7, label="Train Target")
    plt.hist(val_df["target_days_until_next_purchase"].clip(upper=100), bins=35, color="#e17055", alpha=0.7, label="Val Target")
    plt.title("Target Distribution: Days Until Next Purchase (Clipped to 100d)", fontsize=13, pad=12)
    plt.xlabel("Target Days Until Next Purchase", fontsize=11)
    plt.ylabel("Count", fontsize=11)
    plt.legend(fontsize=11)
    plt.grid(axis="y", linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.show()
else:
    print(train_df["target_days_until_next_purchase"].describe())


In [ ]:
# Inspect Phase 3 Quality Report JSON
report_json_path = find_file("data/refillcare/processed/phase3_quality_report.json")
with open(report_json_path, "r") as f:
    report = json.load(f)

print("=== Historical Median Baseline Benchmark ===")
print(f"Fallback Median: {report['baseline_evaluation']['fallback_median_days']} days")
print(f"Validation MAE:  {report['baseline_evaluation']['validation']['mae']} days")
print(f"Validation Acc (within +-7d): {report['baseline_evaluation']['validation']['within_7_days_pct']}%")


## 5. Architectural Findings
- The baseline model achieves **19.20 days MAE** and **40.20% accuracy within a $\pm 7$-day window** on the validation set.
- Machine learning models in Phase 4 must beat this baseline by learning individualized non-linear patterns across history, seasonality, and medicine categories.


## 6. Conclusion
Phase 3 established a leak-free 156K-row training dataset partitioned chronologically, setting a solid foundation for ML model training.
